# 개츠비 1·3장 — 학습 후 Gemma 4 31B 번역 (A100 80GB)

07 노트북에서 학습한 어댑터(`adapter_best.zip`)를 **원본(bf16) 모델에 합쳐서** 번역한다.
설정은 학습 전 비교 대상인 **ctx5와 같음**(앞 문단 5개, 생각 끔, greedy, 개츠비 번역 지침) → 결과를 Claude에게 올리면 학습 전후 6축 비교

**올릴 파일** (1번 셀 실행 시 파일 선택 창에서 세 개 함께)
- `gemma_translate.py` (새 버전), `sentences.jsonl`
- `adapter_best.zip` — 개인 드라이브 `bridgecore_mt/runs/gemma_lora_v1/`에서 내려받은 것

**실행**: ① 설치 셀 → '런타임 → 세션 다시 시작' → 1번 셀부터 끝까지
**시간(추정)**: 모델 불러오기·합치기 5~10분 + 번역 35~45분
**결과**: 마지막 셀이 끝나면 왼쪽 파일 창 `/content/gemma/out/`에서 `gemma_ch1_3_bf16_lora_v1_ctx5.jsonl`, `run_info_lora_v1_ctx5.json`(있으면 `raw_fail_*`도) 내려받기
(자동 다운로드는 셀이 실행 중이면 밀릴 수 있어 파일 창에서 직접 받는 게 확실함)


In [ ]:
# ① 맨 먼저 이 셀만 실행 → 끝나면 '런타임 → 세션 다시 시작' → 다음 셀부터
!pip list --format=freeze | grep -E '^(torch|torchvision|torchaudio)==' > /content/torch_pin.txt
!pip -q install -c /content/torch_pin.txt -U "transformers>=5.10.1" accelerate peft
!pip -q uninstall -y torchao   # 옛 torchao가 있으면 peft가 어댑터를 못 불러옴
!pip list --format=freeze | grep -E '^(transformers|torch|accelerate)=='


## 1. 설정·파일 올리기

In [ ]:
import json, sys, time, platform, inspect
from pathlib import Path
import torch
from google.colab import files

CFG = dict(n_ctx=5, thinking=False)          # 학습 전 비교 대상(ctx5)과 같은 설정
NAME = 'lora_v1_ctx5'
CHAPTERS = ['I', 'III']
MODEL_ID = 'google/gemma-4-31B-it'
BS = 4
THINK_BUDGET = 4096        # 생각 모드일 때 추가로 허용할 토큰 수

BASE = Path('/content/gemma'); OUT_DIR = BASE / 'out'; OUT_DIR.mkdir(parents=True, exist_ok=True)
print('파일 선택 창에서 고르기: gemma_translate.py, sentences.jsonl, adapter_best.zip')
for name, data in files.upload().items():
    clean = Path(name).name.split(' (')[0]
    (OUT_DIR if clean.startswith('gemma_ch1_3') else BASE).joinpath(clean).write_bytes(data)

GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(torch.cuda.get_device_name(0), f'{GPU_GB:.0f}GB')
assert GPU_GB > 70, '80GB GPU가 아닙니다 — 학습 전 번역과 같은 원본(bf16)이어야 비교가 됩니다'
assert (BASE / 'sentences.jsonl').exists() and (BASE / 'gemma_translate.py').exists(), '필수 파일 두 개를 올리세요'
sys.path.insert(0, str(BASE))
import gemma_translate as gt
assert 'n_ctx' in inspect.signature(gt.user_text).parameters, 'gemma_translate.py가 옛 버전입니다 — 새 파일을 올리세요'
import zipfile
ADAPTER = BASE / 'adapter_best'
assert (BASE / 'adapter_best.zip').exists(), 'adapter_best.zip을 올리세요 (드라이브 runs/gemma_lora_v1/)'
zipfile.ZipFile(BASE / 'adapter_best.zip').extractall(ADAPTER)
print('어댑터 파일', sorted(p.name for p in ADAPTER.iterdir()))
assert (ADAPTER / 'adapter_config.json').exists(), '어댑터 zip 안에 adapter_config.json이 없습니다'
print('프롬프트', gt.PROMPT_HASH)


## 2. 모델 불러오기 + 어댑터 합치기 (5~10분)

In [ ]:
from google.colab import userdata
from transformers import AutoProcessor, AutoModelForCausalLM
import transformers
from getpass import getpass

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None   # 공용 계정이면 열쇠 아이콘에 저장하지 말고, 필요할 때만 아래 줄 주석을 풀어 입력
# HF_TOKEN = getpass('HF 토큰: ')

t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
processor.tokenizer.padding_side = 'left'
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map='cuda', token=HF_TOKEN)
from peft import PeftModel
model = PeftModel.from_pretrained(model, str(ADAPTER)).merge_and_unload().eval()   # 어댑터를 원본에 합침(번역 속도 유지)
ADAPTER_CFG = json.loads((ADAPTER / 'adapter_config.json').read_text())
LOAD_SEC = time.time() - t0
print(f'불러오기 {LOAD_SEC/60:.1f}분 · GPU {torch.cuda.memory_allocated()/1e9:.1f}GB')

rows = [json.loads(l) for l in open(BASE / 'sentences.jsonl', encoding='utf-8') if l.strip()]
allb, idx = gt.blocks(rows, CHAPTERS)
WANT = {s['id'] for k in idx for s in allb[k]['sents'] if s['translate']}
print('문단', len(idx), '· 문장', len(WANT))


## 3. 번역 함수 + 시험 1문단

In [ ]:
SAMPLE = dict(do_sample=True, temperature=1.0, top_p=0.95, top_k=64)   # 공식 권장값 — greedy가 실패한 문단 재시도에만 씀

def prompt_text(k, cfg):
    sents, user = gt.user_text(allb, k, n_ctx=cfg['n_ctx'])
    msgs = [{'role': 'system', 'content': gt.SYSTEM + gt.FORMAT}, {'role': 'user', 'content': user}]
    return sents, processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                                enable_thinking=cfg['thinking'])

def split_answer(ids):
    raw = processor.decode(ids, skip_special_tokens=False)
    try:
        r = processor.parse_response(raw)
        return raw, r.get('content') or '', r.get('thinking') or ''
    except Exception:
        return raw, processor.decode(ids, skip_special_tokens=True), ''

@torch.inference_mode()
def generate(ks, cfg, sample=False):
    packs = [prompt_text(k, cfg) for k in ks]
    enc = processor(text=[p[1] for p in packs], return_tensors='pt', padding=True).to(model.device)
    n_in = enc['input_ids'].shape[1]
    need = max(len(p[0]) for p in packs) * 200 + 300 + (THINK_BUDGET if cfg['thinking'] else 0)
    out = model.generate(**enc, max_new_tokens=need, **(SAMPLE if sample else dict(do_sample=False)))
    res = []
    for (sents, _), ids in zip(packs, out[:, n_in:]):
        raw, content, think = split_answer(ids)
        n_tok = int((ids != processor.tokenizer.pad_token_id).sum())
        try:
            got = gt.parse(content)
            miss = [s['id'] for s in sents if not got.get(s['id'])]
            err = f'id 누락 {miss}' if miss else None
        except Exception as e:
            got, err = {}, f'파싱 실패 {e.__class__.__name__}'
        res.append(dict(sents=sents, got=got, err=err, raw=raw, think_chars=len(think), out_tokens=n_tok))
    return res

def run_config(name, cfg, out_path, only=None):
    meta = {'model': MODEL_ID, 'adapter': 'gemma_lora_v1 (07, QLoRA nf4 학습 → bf16에 합침)',
            'adapter_r': ADAPTER_CFG.get('r'), 'runtime': 'colab_transformers_bf16', 'prompt_hash': gt.PROMPT_HASH, 'config': name,
            'n_ctx': cfg['n_ctx'], 'thinking': cfg['thinking'], 'decoding': 'greedy',
            'transformers': transformers.__version__, 'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0)}
    done = {json.loads(l)['id'] for l in open(out_path, encoding='utf-8')} if out_path.exists() else set()
    todo = [k for k in (only if only is not None else idx)
            if any(s['translate'] and s['id'] not in done for s in allb[k]['sents'])]
    order = sorted(todo, key=lambda k: len(gt.user_text(allb, k, n_ctx=cfg['n_ctx'])[1]))
    stats = {'think_chars': [], 'out_tokens': [], 'retried_with_sampling': [], 'fails': {}}
    raw_log = open(OUT_DIR / f'raw_fail_{name}.jsonl', 'a', encoding='utf-8')

    def save(r, sampled=False):
        with open(out_path, 'a', encoding='utf-8') as f:
            for s in r['sents']:
                if s['id'] in done:      # 이미 있는 문장은 다시 쓰지 않음(중복 방지)
                    continue
                f.write(json.dumps({'id': s['id'], 'en': s['en'], 'mt': r['got'][s['id']],
                                    'mt_system': f'gemma-4-31b-it-bf16-{name}',
                                    'mt_meta': {**meta, 'decoding': 'sampling_retry' if sampled else 'greedy'}},
                                   ensure_ascii=False) + '\n')

    t0, bs, i, fails, nb = time.time(), BS, 0, [], 0
    print(f'[{name}] 문단 {len(order)}개 시작')
    while i < len(order):
        ks = order[i:i + bs]
        try:
            res = generate(ks, cfg)
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs == 1:
                raise
            bs = max(1, bs // 2); print('  메모리 부족 → 묶음', bs); continue
        for k, r in zip(ks, res):
            stats['think_chars'].append(r['think_chars']); stats['out_tokens'].append(r['out_tokens'])
            if r['err']:
                fails.append(k); raw_log.write(json.dumps({'k': k, 'err': r['err'], 'raw': r['raw']}, ensure_ascii=False) + '\n')
            else:
                save(r)
        i += len(ks); nb += 1
        if nb % 5 == 0 or i >= len(order):
            el = time.time() - t0
            print(f'  {i}/{len(order)} 문단 · {el/60:.1f}분 · 남은 예상 {el/i*(len(order)-i)/60:.0f}분 · 실패 {len(fails)}')
    for k in fails:                      # greedy로 실패한 문단: 하나씩, 공식 권장 샘플링으로 한 번 더
        r = generate([k], cfg, sample=True)[0]
        if r['err']:
            stats['fails'][str(k)] = r['err']
            raw_log.write(json.dumps({'k': k, 'err': r['err'], 'raw': r['raw'], 'sampling': True}, ensure_ascii=False) + '\n')
        else:
            save(r, sampled=True); stats['retried_with_sampling'].append(k)
    raw_log.close()
    got_ids = {json.loads(l)['id'] for l in open(out_path, encoding='utf-8')} if out_path.exists() else set()
    tc = [x for x in stats['think_chars'] if x]
    info = {**meta, 'gen_sec': round(time.time() - t0), 'n_sentences': len(got_ids & WANT), 'missing': sorted(WANT - got_ids),
            'retried_with_sampling': stats['retried_with_sampling'], 'fails': stats['fails'], 'batch_final': bs,
            'mean_out_tokens': round(sum(stats['out_tokens']) / max(1, len(stats['out_tokens']))),
            'mean_think_chars': round(sum(tc) / len(tc)) if tc else 0, 'load_sec': round(LOAD_SEC),
            'python': platform.python_version()}
    (OUT_DIR / f'run_info_{name}.json').write_text(json.dumps(info, ensure_ascii=False, indent=1))
    print(f'[{name}] 완료 {info["gen_sec"]/60:.1f}분 · {info["n_sentences"]}/{len(WANT)}문장 · 빠짐 {len(info["missing"])}'
          f' · 샘플링 재시도 성공 {len(stats["retried_with_sampling"])} · 평균 생각 {info["mean_think_chars"]}자')
    return info

# 시험 1문단 — 학습 후에도 형식(JSON)이 유지되는지 확인
r = generate(idx[:1], CFG)[0]
print('오류', r['err'], '· 생각', r['think_chars'], '자 · 출력 토큰', r['out_tokens'])
for s in r['sents']:
    print('EN', s['en']); print('KO', r['got'].get(s['id']))
print('--- 원출력 앞부분 ---'); print(r['raw'][:800])


## 4. 전체 번역

In [ ]:
out_path = OUT_DIR / f'gemma_ch1_3_bf16_{NAME}.jsonl'
info = run_config(NAME, CFG, out_path)
print('결과:', out_path)
for f in (out_path, OUT_DIR / f'run_info_{NAME}.json', OUT_DIR / f'raw_fail_{NAME}.jsonl'):
    if f.exists() and f.stat().st_size:
        files.download(str(f))
